# Panou de control - antrenare si testare modeleAici antrenezi, compari si testezi toate modelele, cu parametri pe care ii modifici tu.Fiecare celula de antrenare are un bloc de **parametri** la inceput. Schimba-i,ruleaza celula din nou, si compara in tabelul de la sectiunea 5.Ruleaza celulele in ordine prima data. Dupa aceea poti reveni la oricare.

In [ ]:
import os, sys, time, warningsif os.path.basename(os.getcwd()) == "notebooks":    os.chdir("..")sys.path.insert(0, os.getcwd())warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport torchfrom src.models.evaluate import incarca_date, metrici, tabel_rezultatefrom src.features.build_features import creeaza_apartament, incarca_encoderX_train, X_test, y_train, y_test = incarca_date()encoder = incarca_encoder()rezultate = {}   # aici se aduna toate modelele antrenate in acest notebookmodele = {}print(f"Train: {X_train.shape} | Test: {X_test.shape}")print(f"Encoder: {len(encoder['enc']['oras'])} orase")

## 1. Regresie liniara

In [ ]:
from sklearn.linear_model import LinearRegression, Ridgefrom sklearn.pipeline import make_pipelinefrom sklearn.preprocessing import StandardScaler# ---- parametri ----ALPHA_RIDGE = 1.0# -------------------liniar = LinearRegression().fit(X_train, y_train)modele["Liniar"] = liniarrezultate["Regresie liniara"] = metrici(y_test, liniar.predict(X_test))ridge = make_pipeline(StandardScaler(), Ridge(alpha=ALPHA_RIDGE)).fit(X_train, y_train)modele["Ridge"] = ridgerezultate[f"Ridge (alpha={ALPHA_RIDGE})"] = metrici(y_test, ridge.predict(X_test))print(tabel_rezultate({k: v for k, v in rezultate.items() if "iniar" in k or "Ridge" in k}).to_string())

In [ ]:
# Coeficientii - ce a invatat modelul liniarcoef = pd.Series(liniar.coef_, index=X_train.columns)principale = coef[[c for c in coef.index if not c.startswith("jud_")]]pd.DataFrame({    "coeficient": principale.round(4),    "efect %": (100 * (np.exp(principale) - 1)).round(2),}).sort_values("coeficient", key=abs, ascending=False)

## 2. MLP scikit-learnModifica `ARHITECTURA` si ceilalti parametri, apoi ruleaza din nou.Incearca `(128, 64)`, `(32,)`, `(64, 64, 32)` si vezi ce se schimba.

In [ ]:
from sklearn.neural_network import MLPRegressor# ---- parametri ----ARHITECTURA = (64, 32)ALPHA       = 1e-4      # regularizare L2RATA        = 1e-3BATCH       = 256RABDARE     = 15        # early stoppingSCALARE     = False     # True/False - vezi experimentul din raportSEED        = 42# -------------------mlp = MLPRegressor(hidden_layer_sizes=ARHITECTURA, alpha=ALPHA,                   learning_rate_init=RATA, batch_size=BATCH,                   max_iter=800, early_stopping=True, validation_fraction=0.1,                   n_iter_no_change=RABDARE, random_state=SEED)model_sk = make_pipeline(StandardScaler(), mlp) if SCALARE else mlpt0 = time.time()model_sk.fit(X_train, y_train)print(f"Antrenat in {time.time()-t0:.0f}s, {mlp.n_iter_} epoci")eticheta = f"MLP sklearn {ARHITECTURA}" + (" scalat" if SCALARE else "")modele[eticheta] = model_skrezultate[eticheta] = metrici(y_test, model_sk.predict(X_test))for k, v in rezultate[eticheta].items():    print(f"  {k:14s} {v:,.3f}")

In [ ]:
# Curba de invatare a MLP-ului sklearnplt.figure(figsize=(8, 4))plt.plot(mlp.loss_curve_, label="antrenare")if hasattr(mlp, "validation_scores_") and mlp.validation_scores_:    plt.plot([-s for s in mlp.validation_scores_], label="validare (-R2)")plt.yscale("log"); plt.xlabel("Epoca"); plt.ylabel("Loss")plt.title(f"MLP sklearn {ARHITECTURA}"); plt.legend(); plt.grid(alpha=0.3)plt.show()

## 3. MLP PyTorchAceiasi parametri, dar cu bucla de antrenare scrisa explicit.Setam constantele modulului inainte de antrenare - functiile le citesc de acolo.

In [ ]:
import src.models.mlp_torch as T# ---- parametri ----T.STRATURI        = (64, 32)T.RATA_INVATARE   = 1e-3T.MARIME_BATCH    = 256T.EPOCI_MAX       = 800T.RABDARE         = 15T.WEIGHT_DECAY    = 1e-4# -------------------T.fixeaza_seed(42)X_tr, y_tr, X_val, y_val = T.pregateste_tensori(X_train, y_train)model_torch = T.RetaPreturi(X_train.shape[1], straturi=T.STRATURI)print(model_torch)print(f"Parametri: {sum(p.numel() for p in model_torch.parameters()):,}")t0 = time.time()istoric = T.antreneaza(model_torch, X_tr, y_tr, X_val, y_val, verbose=False)print(f"Antrenat in {time.time()-t0:.0f}s, {len(istoric['train'])} epoci")eticheta_t = f"MLP PyTorch {T.STRATURI}"modele[eticheta_t] = model_torchrezultate[eticheta_t] = metrici(y_test, T.prezice(model_torch, X_test))for k, v in rezultate[eticheta_t].items():    print(f"  {k:14s} {v:,.3f}")

In [ ]:
# Curba de invatare PyTorchplt.figure(figsize=(8, 4))plt.plot(istoric["train"], label="antrenare")plt.plot(istoric["validare"], label="validare")plt.axvline(int(np.argmin(istoric["validare"])), color="crimson", ls="--",            label="cel mai bun model")plt.yscale("log"); plt.xlabel("Epoca"); plt.ylabel("MSE pe log(pret)")plt.title(f"PyTorch {T.STRATURI}"); plt.legend(); plt.grid(alpha=0.3)plt.show()

## 4. Random Forest (optional - al treilea model)

In [ ]:
from sklearn.ensemble import RandomForestRegressor# ---- parametri ----N_ARBORI   = 300ADANCIME   = None      # None = fara limitaMIN_FRUNZA = 2# -------------------rf = RandomForestRegressor(n_estimators=N_ARBORI, max_depth=ADANCIME,                           min_samples_leaf=MIN_FRUNZA, n_jobs=-1, random_state=42)t0 = time.time()rf.fit(X_train, y_train)print(f"Antrenat in {time.time()-t0:.0f}s")modele["Random Forest"] = rfrezultate[f"Random Forest ({N_ARBORI})"] = metrici(y_test, rf.predict(X_test))for k, v in rezultate[f"Random Forest ({N_ARBORI})"].items():    print(f"  {k:14s} {v:,.3f}")

In [ ]:
# Ce features conteaza pentru Random Forestimp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)imp.head(12).plot.barh(figsize=(7, 4.5)).invert_yaxis()plt.title("Importanta features - Random Forest"); plt.xlabel("importanta")plt.tight_layout(); plt.show()

## 5. Comparatia tuturor modelelor antrenate aici

In [ ]:
tabel = tabel_rezultate(rezultate)tabel_sortat = tabel.loc[pd.DataFrame(rezultate).T["MAE (EUR)"].sort_values().index]tabel_sortat

In [ ]:
# Salveaza in tabelul cumulativ din reports/ (optional)from src.models.evaluate import salveaza_rezultatesalveaza_rezultate(rezultate)

## 6. Analiza de sensibilitate - orice model, orice featureSchimbi un singur feature si vezi cum reactioneaza fiecare model.Asta arata *ce a invatat* modelul, nu doar cat de bine punteaza.

In [ ]:
def prezice_orice(model, X):    """Predictii in spatiul log, indiferent daca modelul e sklearn sau torch."""    if isinstance(model, torch.nn.Module):        return T.prezice(model, X)    return model.predict(X)def sensibilitate(feature, valori, n_esantion=300, seed=42):    """Media predictiilor cand fortam un feature la fiecare valoare din lista."""    esantion = X_test.sample(n_esantion, random_state=seed)    out = {}    for nume, model in modele.items():        linie = {}        for val in valori:            v = esantion.copy()            v[feature] = val            if feature == "etaj":                v["etaj_lipsa"] = 0            if feature == "an_constructie_ord":                v["an_lipsa"] = 0            linie[val] = np.exp(prezice_orice(model, v)).mean()        out[nume] = pd.Series(linie)    return pd.DataFrame(out)

In [ ]:
# ---- ce testam ----FEATURE = "etaj"VALORI  = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]# -------------------s = sensibilitate(FEATURE, VALORI)relativ = 100 * (s / s.iloc[0] - 1)fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))s.plot(marker="o", ax=axes[0]); axes[0].set_title(f"Pret prezis vs. {FEATURE}")axes[0].set_xlabel(FEATURE); axes[0].set_ylabel("EUR")relativ.plot(marker="o", ax=axes[1]); axes[1].axhline(0, color="black", lw=1)axes[1].set_title("Variatie relativa (%)"); axes[1].set_xlabel(FEATURE)plt.tight_layout(); plt.show()relativ.round(1)

In [ ]:
# Alte feature-uri de incercat - decomenteaza si ruleaza celula de mai sus cu ele# FEATURE = "an_constructie_ord"; VALORI = [0, 1, 2, 3]# FEATURE = "camere";            VALORI = [1, 2, 3, 4]# FEATURE = "log_suprafata";     VALORI = list(np.log([30, 45, 60, 80, 100, 140]))sensibilitate("an_constructie_ord", [0, 1, 2, 3]).round(0)

## 7. Estimeaza pretul unui apartament inventatSchimba valorile si vezi ce zic toate modelele.

In [ ]:
# ---- apartamentul tau ----ORAS       = "Cluj-Napoca"SUPRAFATA  = 65CAMERE     = 2ETAJ       = 3AN         = 3        # 0=inainte de 1977, 1=1977-1990, 2=1990-2000, 3=dupa 2000AGENTIE    = False# --------------------------ap = creeaza_apartament(ORAS, suprafata_mp=SUPRAFATA, camere=CAMERE, etaj=ETAJ,                        an_ord=AN, agentie=AGENTIE, encoder=encoder)print(f"{SUPRAFATA} mp, {CAMERE} camere, etaj {ETAJ}, {ORAS}")print()for nume, model in modele.items():    pret = np.exp(prezice_orice(model, ap))[0]    print(f"  {nume:26s} {pret:10,.0f} EUR   ({pret/SUPRAFATA:6,.0f} EUR/mp)")

In [ ]:
# Acelasi apartament, in mai multe oraseORASE = ["Cluj-Napoca", "Bucuresti", "Brasov", "Timisoara", "Iasi",         "Floresti", "Craiova", "Arad", "Braila", "Vaslui"]randuri = []for oras in ORASE:    try:        ap = creeaza_apartament(oras, suprafata_mp=SUPRAFATA, camere=CAMERE,                                etaj=ETAJ, an_ord=AN, encoder=encoder)        randuri.append({nume: np.exp(prezice_orice(m, ap))[0] for nume, m in modele.items()}                       | {"oras": oras})    except ValueError as e:        print(e)pd.DataFrame(randuri).set_index("oras").round(0)

## Idei de experimente- **Arhitectura**: `(32,)` vs `(64, 32)` vs `(128, 64, 32)` - creste performanta cu adancimea?- **Regularizare**: `ALPHA` de la `1e-5` la `1e-1` - unde incepe sa strice?- **Rata de invatare**: `1e-2` diverge? `1e-4` invata prea incet?- **Random Forest**: `ADANCIME = 10` vs `None` - cat de mult conteaza limitarea?- **Sensibilitate**: exista vreun feature unde liniarul si reteaua chiar sunt de acord?Fiecare rulare se adauga in `rezultate`, deci tabelul de la sectiunea 5 creste singur.